# Solution 2.2.3 — Cleaning, Missing Values & Duplicates

### Path Setup

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
clean_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

# Typed checkpoint from 2.2.2 — income is numeric, dates parsed, category typos fixed
df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded typed checkpoint:', df.shape)
df.head()

---

## Task 1 — Never destroy the input: make a clean copy

Cleaning operations should never touch the DataFrame you loaded. Create a separate working copy and apply every change to it. This keeps the checkpoint intact for comparison and lets you restart if your assumptions change.

In [ ]:
df_clean = df.copy()

print('Working copy:', df_clean.shape)

All cleaning below is applied to `df_clean`. The loaded `df` stays untouched so we can compare before/after at any point and restart if an assumption changes.

---

## Task 2 — Detect missing values

`isna().sum()` counts blanks/`NaN` per column. The percentage version shows how serious each gap is, and inspecting the affected rows shows *which* records are incomplete. Because 2.2.2 already converted `income_dkw` and `survey_date`, their text placeholders (`unknown`, `not recorded`) are already real `NaN`/`NaT` here.

In [ ]:
df_clean.isna().sum()

In [ ]:
(df_clean.isna().sum() / len(df_clean) * 100).round(2)

In [ ]:
df_clean[df_clean.isna().any(axis=1)]

**Answers:**

- At this stage the gaps are: `region_code`, `province_name`, `income_dkw`, and `survey_date`. Because 2.2.2 already converted income and dates, the old text placeholders (`unknown`, `not recorded`) are already counted as `NaN`/`NaT` here.
- The single-cell gaps in `province_name` and `region_code` are minor; the `income_dkw` gaps matter most because income is a key analysis variable.

---

## Task 3 — Coded missing values

Survey data mixes a few problems that all look like numbers:

| Value | Meaning | Fix |
|---|---|---|
| `-5000` | negative income — a **sign-entry error** | recover with `.abs()` |
| `999999` | all-nines "not stated" **sentinel** | recode to `NaN` |
| `999`, `9999`, `99` | column sentinels (age, density, education) | recode to `NaN` |

Not every odd value is missing: a negative income is a fixable typo whose magnitude is real, while a sentinel carries no real value. Handle each appropriately and check the impact.

In [ ]:
# A negative income is a sign-entry error — recover the magnitude (it is NOT missing)
df_clean['income_dkw'] = df_clean['income_dkw'].abs()

# 999999 is an all-nines "not stated" sentinel — recode it to NaN
df_clean['income_dkw'] = df_clean['income_dkw'].replace(999999, np.nan)

df_clean['income_dkw'].describe()

In [ ]:
def recode_coded_missing(series, codes):
    """Replace coded missing values with NaN."""
    return series.replace(codes, np.nan)

print('Mean age BEFORE recode:', round(df_clean['age'].mean(), 1))

df_clean['age'] = recode_coded_missing(df_clean['age'], [999])
df_clean['pop_density'] = recode_coded_missing(df_clean['pop_density'], [9999])
df_clean['education_code'] = recode_coded_missing(df_clean['education_code'], [99])

print('Mean age AFTER recode: ', round(df_clean['age'].mean(), 1))

**Answers:**

- A negative income is almost always a data-entry **sign error**: the magnitude (`5000`) is real, so `.abs()` recovers the record instead of throwing it away. `999999` carries no real magnitude — it is a placeholder for "not stated", so it must become `NaN`.
- A single `999` drags the mean age up by years; leaving sentinels in place silently corrupts every statistic that touches the column.
- `0` is tricky because it can be a *real* value (a household with no recorded income) or a "not collected" placeholder — you cannot tell from the number alone, so check the survey documentation before treating `0` as missing.

---

## Task 4 — Missing-value strategy: drop critical-only

Three strategies exist: drop any row with a gap, drop only rows missing a **critical** column, or **fill** the gaps. Dropping everything is usually too aggressive. Here the only non-negotiable column is the identifier `hh_id` — drop rows missing it, leave the rest.

In [ ]:
print('Before:', df_clean.shape)
df_clean = df_clean.dropna(subset=['hh_id'])
print('After: ', df_clean.shape)

In [ ]:
example = df_clean['income_dkw'].fillna(df_clean['income_dkw'].median())
print('NaN before fill:', df_clean['income_dkw'].isna().sum(),
      '| NaN after fill:', example.isna().sum())

**Answers:**

- No rows were dropped — every record has an `hh_id`. The check is still worth running: it documents the assumption that the identifier is mandatory and would protect you on a dirtier extract.
- Median-filling income is acceptable when values are *missing at random* and you need a complete column for a model. It biases results when the gaps are systematic (e.g. richer households refuse to answer), so we leave income honestly `NaN` here.

---

## Task 5 — Detect and remove duplicates

Real surveys record the same household twice. First handle **exact** duplicates (every column identical), then **subset** duplicates (same `hh_id`, different other fields).

In [ ]:
print('Exact duplicate rows:', df_clean.duplicated().sum())
df_clean[df_clean.duplicated(keep=False)].sort_values('hh_id')

In [ ]:
print('Before:', df_clean.shape)
df_clean = df_clean.drop_duplicates()
print('After: ', df_clean.shape)

In [ ]:
dups = df_clean[df_clean.duplicated(subset=['hh_id'], keep=False)]
dups.sort_values('hh_id')[['hh_id', 'district', 'income_dkw', 'survey_date']]

When two rows share an `hh_id` but differ, choose which to keep. A robust rule is **keep the most complete** record (the one with the fewest missing values).

*Quando duas linhas partilham o mesmo `hh_id` mas diferem, escolha qual manter. Uma regra robusta é **manter o registo mais completo** (o que tem menos valores em falta).*


In [ ]:
df_clean['missing_count'] = df_clean.isna().sum(axis=1)
df_clean = (
    df_clean
    .sort_values(['hh_id', 'missing_count'])
    .drop_duplicates(subset=['hh_id'], keep='first')
)
df_clean = df_clean.drop(columns='missing_count')
print('After resolving subset duplicates:', df_clean.shape)

**Answers:**

- `HH0012` was an **exact** duplicate (two identical rows) — `drop_duplicates()` removed one.
- `HH0005` was a **subset** duplicate: two rows with the same id but different `income_dkw` and `survey_date`. One had a missing income (originally `'unknown'`); the *most complete* record (with the real income) was kept. An alternative rule is "keep the most recent": `sort_values('survey_date').drop_duplicates(subset=['hh_id'], keep='last')`.
- `keep=False` shows **all** copies of a duplicate, which is what you want when inspecting; the default `keep='first'` hides the first occurrence.

---

## Task 6 — Validation rules: catch impossible values

Some values are not missing — they are *impossible*. A household cannot have `0`, `-1`, or `99` members. Filter out the rows that fail a plausibility rule, checking the shape before and after.

In [ ]:
print('hh_size values:', sorted(df_clean['hh_size'].unique()))
print('Before:', df_clean.shape)
df_clean = df_clean[df_clean['hh_size'].between(1, 20)]
print('After: ', df_clean.shape)

**Answers:**

- The rule removed 3 rows: `HH0008` (`hh_size` 0), `HH0011` (`-1`), and `HH0013` (`99`). None describe a possible household.
- **Recoding to NaN** keeps the household and just marks one field unknown; **dropping the row** discards the whole record. Drop only when the row is unusable — an impossible household size makes the entire record meaningless.

---

## Task 7 — Save the cleaned dataset

Overwrite the checkpoint in `10_cleaned/` with the fully cleaned result, then reload it to confirm it round-trips. (Raw data in `0_raw/` is never touched.)

In [ ]:
print('Final cleaned shape:', df_clean.shape)
df_clean.isna().sum()

In [ ]:
df_clean = df_clean.reset_index(drop=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_clean.to_csv(out_path, index=False)
print('Saved:', out_path)

In [ ]:
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded shape:', check.shape)
print()
print(check.dtypes)
check.head()

**Answers:**

- Starting from the 28-row checkpoint, 23 rows survive: -1 exact duplicate, -1 subset duplicate, -3 impossible household sizes. (The negative income is recovered with `.abs()`, so no row is dropped for it.)
- `survey_date` reloads as `object` (text) — CSV cannot store a datetime type. Before any date analysis you must convert it again with `pd.to_datetime()`.
- Every excluded row is justifiable: duplicates or impossible household sizes. That audit trail is what makes the cleaning defensible.